In [9]:
import matplotlib.pyplot as plt
import networkx as nx

# from DiffusionTrend import DiffusionTrend
# from SIRVModel import SIRVModel
# from FastSIRV import FastSIRV
# import SIRV_viz as viz
# from Gloabl_stage import GlobalStage
from utils.sweep import run_sweep
# from sweep import get_sirv_core

import numpy as np
import pandas as pd
# import seaborn as sns
from time import time
from pathlib import Path

# from joblib import Parallel, delayed
# import itertools
# import os

%matplotlib inline
plt.rcParams['text.usetex'] = True
# plt.rcParams.update({
#     'text.usetex': True,
#     'font.family': 'sans-serif',
#     'font.sans-serif': ['DejaVu Sans'],
#     'font.weight': 'bold',
#     'mathtext.fontset': 'dejavusans',
#     # 'text.latex.preamble': r'\usepackage{amsmath}'
# })
# plt.rcParams.update({
#     'text.usetex': True,
#     'font.family': 'sans-serif',
#     'font.sans-serif': ['DejaVu Sans'],
#     'font.weight': 'bold',
#     'mathtext.fontset': 'dejavusans'
# })

p = Path('data')
p

PosixPath('data')

In [3]:
# df = run_sweep(theta_values=[0.5], M=50, N=50, Dr=0.5, Dg=0.5, epsilon=0.01, grid_step=0.1, n_jobs=6, outfile=None)
df = pd.read_csv(p / '100, PD, 0.01.csv')
df

,x,n,payoff_C,payoff_D,epi_size,C,eta,beta,gamma,theta,x0,n0,L,Dr,Dg,Game
0,0.084042,0.000010,-0.009575,-0.009582,0.011414,0.1,0.1,0.1,0.1,0.5,0.5,0.5,100,0.5,0.5,PD
1,0.045183,0.000161,-0.004658,-0.005415,0.006667,0.1,0.1,0.1,0.2,0.5,0.5,0.5,100,0.5,0.5,PD
2,0.041697,0.000183,-0.004385,-0.003765,0.005556,0.1,0.1,0.1,0.3,0.5,0.5,0.5,100,0.5,0.5,PD
3,0.040258,0.000173,-0.004063,-0.004082,0.005455,0.1,0.1,0.1,0.4,0.5,0.5,0.5,100,0.5,0.5,PD
4,0.039551,0.000181,-0.003992,-0.003363,0.004545,0.1,0.1,0.1,0.5,0.5,0.5,0.5,100,0.5,0.5,PD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.077489,0.000142,-0.078260,-0.078691,0.086263,1.0,1.0,1.0,0.6,0.5,0.5,0.5,100,0.5,0.5,PD
9996,0.058620,0.000161,-0.059172,-0.059364,0.066667,1.0,1.0,1.0,0.7,0.5,0.5,0.5,100,0.5,0.5,PD
9997,0.046177,0.000132,-0.046654,-0.047282,0.057475,1.0,1.0,1.0,0.8,0.5,0.5,0.5,100,0.5,0.5,PD
9998,0.038755,0.000178,-0.039177,-0.038423,0.045859,1.0,1.0,1.0,0.9,0.5,0.5,0.5,100,0.5,0.5,PD


In [ ]:
import matplotlib.lines as mlines

def get_double_plot_fast(data, col='x', game_type='H',
                         max_outer=None,         # 只畫前幾個外層 (gamma, beta)，例如 (3,3)
                         downsample=1,           # 內層 (eta, C) 每隔幾格取一次
                         cmap='jet'):
    data = data[data['Game'] == game_type]
    
    # ---------- 1) 取唯一值 (避免排序開銷過多，必要時才排序) ----------
    outer_x = np.sort(data['beta'].unique())                  # β 升序
    outer_y = np.sort(data['gamma'].unique())[::-1]           # γ 降序
    inner_x = np.sort(data['C'].unique())                     # C
    inner_y = np.sort(data['eta'].unique())                   # η

    B, G, Cn, En = len(outer_x), len(outer_y), len(inner_x), len(inner_y)

    # ---------- 2) 把 (β,γ,η,C) 映射到 4D array 位置，避開 pivot ----------
    # 建類別編碼（O(n)）
    bi = pd.Categorical(data['beta'], categories=outer_x, ordered=True).codes
    gi = pd.Categorical(data['gamma'], categories=outer_y, ordered=True).codes
    ci = pd.Categorical(data['C'],    categories=inner_x, ordered=True).codes
    ei = pd.Categorical(data['eta'],  categories=inner_y, ordered=True).codes

    arr = np.full((G, B, En, Cn), np.nan, dtype=float)
    vals = data[col].to_numpy()
    arr[gi, bi, ei, ci] = vals   # 直接填值

    # ---------- 3) 色階範圍 ----------
    if col == 'payoff_D':
        vmin, vmax = -1.0, 0.0
    elif col == 'payoff_C':
        vmin, vmax = -1.0, 1.0
    else:
        vmin, vmax = 0.0, 1.0

    # ---------- 4) 限縮繪製區域 + 降採樣 ----------
    g_lim = G if not max_outer else min(G, max_outer[0])
    b_lim = B if not max_outer else min(B, max_outer[1])
    # 內層降採樣（為了看 layout/字體很夠用）
    arr_view = arr[:g_lim, :b_lim, ::downsample, ::downsample]
    inner_y_ticks = inner_y[::downsample]
    inner_x_ticks = inner_x[::downsample]

    # ---------- 5) 畫圖：用 imshow（快），關掉多餘 ticks ----------
    fig, axes = plt.subplots(nrows=g_lim, ncols=b_lim,
                             figsize=(20, 20), sharex=False, sharey=False)
    if g_lim == 1 and b_lim == 1:
        axes = np.array([[axes]])
    elif g_lim == 1:
        axes = axes[np.newaxis, :]
    elif b_lim == 1:
        axes = axes[:, np.newaxis]

    fig.subplots_adjust(wspace=0.1, hspace=0.1)

    for i in range(g_lim):
        for j in range(b_lim):
            ax = axes[i, j]
            # 注意：原本你用 [::-1] 讓 η 反序，本質是把 row 上下翻轉。
            # imshow 用 origin='upper' 就能符合「上方是大的 η」的直覺，不需額外複製數據。
            im = ax.imshow(arr_view[i, j], vmin=vmin, vmax=vmax, cmap=cmap,
                           origin='lower', aspect='auto', interpolation='nearest',
                           rasterized=True)

            # 只有左下角保留 tick（且只標 0.1 / 1 的兩端）
            if i != g_lim - 1 and j == 0 and (i % 3 == 0):
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_ylabel('')
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels(["", ""], fontsize=38)
            elif i == g_lim - 1 and j != 0 and (j % 3 == 0):
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_xlabel('')
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels(["", ""], fontsize=38)
            elif i == g_lim - 1 and j == 0:
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_ylabel('')
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels([r"$0.1$", r"$1.0$"], fontsize=38)
                ax.set_xlabel('')
            else:
                ax.set_xticks([0, arr_view.shape[-1]-1])
                ax.set_xticklabels(["", ""], fontsize=38)
                ax.set_yticks([0, arr_view.shape[-2]-1])
                ax.set_yticklabels(["", ""], fontsize=38)

    # ---------- 6) 外層標籤 + 單一 colorbar ----------
    fig.text(-4.5, -1.5, r'$\beta$', ha='center', va='center', fontsize=96, transform=ax.transAxes)
    fig.text(-11.5, 5.5, r'$\gamma$', ha='center', va='center', rotation=0, fontsize=96, transform=ax.transAxes)
    fig.text(-5, -0.5, r'$C$', ha='center', va='center', fontsize=64, transform=ax.transAxes)
    fig.text(-10.5, 5, r'$\eta$', ha='center', va='center', fontsize=64, transform=ax.transAxes)

    # fig.text(0.16, 0.04, r'$0.1$', ha='center', va='center', fontsize=48)
    # fig.text(0.385, 0.04, r'$0.4$', ha='center', va='center', fontsize=48)
    # fig.text(0.61, 0.04, r'$0.7$', ha='center', va='center', fontsize=48)
    # fig.text(0.83, 0.04, r'$1.0$', ha='center', va='center', fontsize=48)

    # fig.text(0.06, 0.14, r'$0.1$', ha='center', va='center', fontsize=48)
    # fig.text(0.06, 0.375, r'$0.4$', ha='center', va='center', fontsize=48)
    # fig.text(0.06, 0.61, r'$0.7$', ha='center', va='center', fontsize=48)
    # fig.text(0.06, 0.84, r'$1.0$', ha='center', va='center', fontsize=48)

    if col == 'x':
        title_dict = {
            'CH': ('a', 'Chicken'), 
            'PD': ('b', 'Prisoner\'s Dilemma'),
            'H': ('c', 'Harmony'),
            'SH': ('d', 'Stag Hunt'),
            }
    else:
        title_dict = {
            'PD': ('a', 'Prisoner\'s Dilemma'),
            'H': ('b', 'Harmony'),
            }

    title = title_dict[game_type]
    # a, b, c, d
    fig.text(-11.4, 11.4, f"({title[0]})", ha='left', va='bottom', fontsize=64, transform=ax.transAxes)
    # Game type
    fig.text(-4.4, 11.2, fr"\bf {title[1]}", ha='center', va='bottom', fontsize=84, transform=ax.transAxes)


    # 共享 colorbar（用最後一個 im 的 mappable）
    boundaries = np.linspace(0, 1, 21)  # 10個區間
    cbar = fig.colorbar(axes[0, 0].images[0], ax=axes, orientation='vertical',
                        boundaries=boundaries,
                        fraction=0.02, aspect=30, pad=0.03)
    tick_positions = np.linspace(0, 1, 6)  # 顯示 5 個主要刻度
    cbar.set_ticks(tick_positions)
    cbar.set_ticklabels([fr'${x:.1f}$' for x in tick_positions])
    cbar.ax.tick_params(labelsize=64, pad=20)

    # === KEY CHANGE: 新增程式碼以繪製指定的外圍刻度 ===
    # ---------- 7) 繪製指定位置的【浮動式】外層刻度線 ---
    outer_tick_len = 0.01  # 刻度線的長度
    outer_tick_gap = 0.02  # 刻度線與圖表邊界的【間距】
    outer_pad = 0.015       # 數字標籤與刻度線的間距

    # --- 外部 X 軸 (beta) 的指定刻度 ---
    target_betas = [0.1, 0.4, 0.7, 1.0]
    for beta_val in target_betas:
        j = np.argmin(np.abs(outer_x - beta_val))
        ax = axes[-1, j]
        pos = ax.get_position()
        x_center = pos.x0 + pos.width / 2
        
        # === KEY CHANGE: 調整線條的 Y 座標 ===
        # 線條的起點和終點都向下平移 outer_tick_gap 的距離
        y_start = pos.y0 - outer_tick_gap
        y_end = pos.y0 - outer_tick_gap - outer_tick_len
        line_bottom = mlines.Line2D([x_center, x_center], [y_start, y_end], 
                                    color='black', transform=fig.transFigure)
        fig.add_artist(line_bottom)
        
        # 文字位置也跟著調整
        fig.text(x_center, y_end - outer_pad, fr"${beta_val:.1f}$", 
                 ha='center', va='top', fontsize=64, transform=fig.transFigure)
                 
    # --- 外部 Y 軸 (gamma) 的指定刻度 ---
    target_gammas = [0.1, 0.4, 0.7, 1.0]
    for gamma_val in target_gammas:
        i = np.argmin(np.abs(outer_y - gamma_val))
        ax = axes[i, 0]
        pos = ax.get_position()
        y_center = pos.y0 + pos.height / 2

        # === KEY CHANGE: 調整線條的 X 座標 ===
        # 線條的起點和終點都向左平移 outer_tick_gap 的距離
        x_start = pos.x0 - outer_tick_gap
        x_end = pos.x0 - outer_tick_gap - outer_tick_len
        line_left = mlines.Line2D([x_start, x_end], [y_center, y_center], 
                                  color='black', transform=fig.transFigure)
        fig.add_artist(line_left)

        # 文字位置也跟著調整
        fig.text(x_end - outer_pad, y_center, fr"${gamma_val:.1f}$", 
                 ha='right', va='center', fontsize=64, transform=fig.transFigure)

    plt.show()
    return

# (a) CH (b) PD
# (c) H  (d) SH

game_type = 'PD'

get_double_plot_fast(df, 'x', game_type)